# ₥ Création de la Base PostgreSQL

### connexion 

In [1]:
from sqlalchemy import create_engine,text
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

print("Connexion réussie")

Connexion réussie


### creation des tables

In [2]:
from sqlalchemy import text

with engine.begin() as conn:
    with open(r"C:\Users\manal\OneDrive\Desktop\Stockage des Données Bancaires\sql\schema.sql", "r", encoding="utf-8") as file:
        sql_script = file.read()

    try:
        conn.execute(text(sql_script))
        print("Schema exécuté avec succès")
    except Exception as e:
        print("Schema déjà existant ou erreur :", e)

Schema exécuté avec succès


In [3]:
'''with engine.begin() as conn:
  with open(r"C:/Users/manal/Desktop/Stockage des Données Bancaires/sql/schema.sql", "r", encoding="utf-8") as file:
        sql_script = file.read()
        conn.execute(text(sql_script))

print("Schema exécuté avec succès")'''

'with engine.begin() as conn:\n  with open(r"C:/Users/manal/Desktop/Stockage des Données Bancaires/sql/schema.sql", "r", encoding="utf-8") as file:\n        sql_script = file.read()\n        conn.execute(text(sql_script))\n\nprint("Schema exécuté avec succès")'

In [4]:
with engine.connect() as conn:
  with open(r"C:\Users\manal\OneDrive\Desktop\Stockage des Données Bancaires\sql\index.sql", "r", encoding="utf-8") as file:
        sql_script = file.read()
        conn.execute(text(sql_script))
        conn.commit()

print("Index crées avec succès")

Index crées avec succès


In [5]:
with engine.connect() as conn:
  with open(r"C:\Users\manal\OneDrive\Desktop\Stockage des Données Bancaires\sql\views.sql", "r", encoding="utf-8") as file:
        sql_script = file.read()
        conn.execute(text(sql_script))
        conn.commit()

print("Views  crées avec succès")

Views  crées avec succès


# Chargement des Données via SQLAlchemy

### Séparer financecore_clean.csv selon les tables normalisées avant insertion

In [6]:
import pandas as pd
df=pd.read_csv("C:/Users/manal/OneDrive/Desktop/Stockage des Données Bancaires/data/financecore_clean.csv")

In [7]:
segments=df[[
         "segment_client" 
]].drop_duplicates()
segments["id_segment"]=range(1, len(segments) + 1)
segments=segments.rename(columns={"segment_client":"nom_segments"})
segments[[
    "id_segment",
    "nom_segments"
]].head()

,id_segment,nom_segments
0,1,Premium
1,2,Risque
2,3,Standard


In [8]:
# Renommage des colonnes
temps = df.rename(columns={
    
    "jour-semaine": "jour_semaine"
})

# Sélection + suppression des doublons sur la vraie clé métier
temps = temps[[
    "date_transaction",
    "annee",
    "mois",
    "trimestre",
    "jour_semaine"
]].drop_duplicates(subset=["date_transaction"])

# Réindexation propre
temps = temps.reset_index(drop=True)

# Création d'un ID propre
temps["id_temps"] = temps.index + 1

# Réorganisation finale des colonnes
temps = temps[[
    "id_temps",
    "date_transaction",
    "annee",
    "mois",
    "trimestre",
    "jour_semaine"
]]
temps['date_transaction'] = pd.to_datetime(temps['date_transaction']).dt.date
temps = temps.drop_duplicates(subset=["date_transaction"])
temps.head()

,id_temps,date_transaction,annee,mois,trimestre,jour_semaine
0,1,2022-04-19,2022,4,2,1
1,2,2024-06-20,2024,6,2,3
2,3,2024-08-28,2024,8,3,2
3,4,2024-01-07,2024,1,1,6
4,5,2024-08-11,2024,8,3,6


In [9]:
clients = df[[
    "client_id",
    "score_credit_client",
    "categorie_risque",
    "taux_rejet",
    "segment_client",
]].drop_duplicates()

clients = clients.rename(columns={"client_id": "id_client"})
clients = clients.rename(columns={
    "score_credit_client": "score_credit",
    "segment_client" : "nom_segments"
})


clients = clients.merge(segments, on="nom_segments", how="left")
clients = clients.drop_duplicates(subset=["id_client"])
clients = clients.drop(columns=["nom_segments"])
clients[[
    "id_client",
    "score_credit",
    "categorie_risque",
    "taux_rejet",
    "id_segment",
]].drop_duplicates().head()

,id_client,score_credit,categorie_risque,taux_rejet,id_segment
0,CLI0060,645.0,Medium,0.055728,1
1,CLI0057,435.0,High,0.045894,2
2,CLI0015,648.0,Medium,0.040134,3
3,CLI0045,704.0,Low,0.045894,3
4,CLI0034,457.0,High,0.045894,2


In [10]:
produits = df[[
    "produit",
    "categorie"
]].drop_duplicates()

produits["id_produit"] = range(1, len(produits) + 1)
produits=produits.rename(columns={"produit" : "nom_produit"})

produits = produits[[
    "id_produit",
    "nom_produit",
    "categorie"
]]
produits.head()

,id_produit,nom_produit,categorie
0,1,Compte Epargne,Depot especes
1,2,Credit Consommation,Retrait DAB
2,3,PEA,Prelevement
3,4,Credit Consommation,Paiement CB
4,5,Credit Immobilier,Interets


In [11]:
comptes=df[[
    "client_id",
    "solde_avant"
]].drop_duplicates()
comptes=comptes.rename(columns={"client_id" : "id_client"})

comptes=comptes.rename(columns={"solde_avant" : "solde"})


comptes["id_compte"]=range(1,len(comptes)+1)

comptes[[ 
    "id_compte",
    "id_client",
    "solde"
    ]].head()

,id_compte,id_client,solde
0,1,CLI0060,16415.10
1,2,CLI0057,42890.81
2,3,CLI0015,48489.38
3,4,CLI0045,43962.51
4,5,CLI0034,17312.83


In [12]:
agences=df[[
    "agence"  
]].drop_duplicates()
agences=agences.rename(columns={"agence" : "nom_agence"})


agences["id_agence"]=range(1,len(agences)+1)
    
agences[[
    "id_agence", 
    "nom_agence" 
 ]].head()
    

,id_agence,nom_agence
0,1,Marseille-Vieux-Port
1,2,Bordeaux-Meriadeck
2,3,Lyon-Part-Dieu
6,4,Lille-Grand-Place
9,5,Toulouse-Capitole


In [13]:
df = df.merge(
    comptes[["id_compte", "id_client"]],
    left_on="client_id",
    right_on="id_client",
    how="left"
)
print("id_compte" in df.columns)

True


In [14]:
transactions = df[[
    "transaction_id",
    "id_compte",
    "produit",
    "agence",
    "date_transaction",
    "montant",
    "devise",
    "montant_eur",
    "type_operation",
    "statut"
]].drop_duplicates()


In [15]:
transactions["date_transaction"] = pd.to_datetime(transactions["date_transaction"]).dt.normalize()
temps["date_transaction"] = pd.to_datetime(temps["date_transaction"]).dt.normalize()

In [16]:
transactions = transactions.merge(
    temps[["id_temps", "date_transaction"]],
    left_on="date_transaction",
    right_on="date_transaction",
    how="left"
)


In [17]:
transactions = transactions.merge(
    produits[["id_produit", "nom_produit"]],
    left_on="produit",
    right_on="nom_produit",
    how="left"
)

In [18]:
transactions = transactions.merge(
    comptes[["id_compte"]],
    how="left"
)

In [19]:
transactions = transactions.merge(
    agences,
    left_on="agence",
    right_on="nom_agence"
)

In [20]:
transactions = transactions.rename(columns={
    "transaction_id": "id_transaction",
    "compte_id": "id_compte"
})
transactions=transactions.drop_duplicates(subset='id_transaction')
transactions = transactions[[
    "id_transaction",
    "id_compte",
    "id_produit",
    "id_agence",
    "id_temps",
    "montant",
    "devise",
    "montant_eur",
    "type_operation",
    "statut"
]].drop_duplicates()
transactions.head()

,id_transaction,id_compte,id_produit,id_agence,id_temps,montant,devise,montant_eur,type_operation,statut
0,TXN000559,1,1,1,1,2050.42,EUR,2050.42,Credit,Complete
96,TXN001154,2,2,2,2,-123.66,GBP,-143.79,Debit,Rejete
168,TXN000764,3,3,3,3,-396.17,EUR,-396.17,Debit,Complete
256,TXN001598,4,2,2,4,225.20,EUR,225.20,Credit,Complete
392,TXN001873,5,5,2,5,935.32,EUR,935.32,Credit,Complete


In [21]:
temps.to_sql("temps", engine, if_exists="append", index=False)

907

In [22]:
segments.to_sql("segments", engine, if_exists="append", index=False)

3

In [23]:
clients.to_sql("clients",engine,if_exists="append",index=False)

150

In [24]:
comptes.to_sql("comptes",engine,if_exists="append",index=False)

1000

In [25]:
produits.to_sql("produits",engine,if_exists="append",index=False)

64

In [26]:
agences.to_sql("agences",engine,if_exists="append",index=False)

8

In [27]:
from sqlalchemy import create_engine


engine = create_engine(os.getenv("DATABASE_URL"))

In [28]:
transactions.to_sql(
    "transactions",
    engine,
    if_exists="append",
    index=False
)

1000